# 🧫 Raman Bioprocess Temporal Monitoring

I am using a real public bioprocess dataset to learn how **spectral signals evolve over time**.

It is penicillin fermentation. It gives me the missing ingredient that many small cartilage spectroscopy datasets do not give me: **time-resolved spectra**.

## Main question

Given Raman/process measurements arriving during a biological production run, can I estimate:

- the current age of the batch in days?
- how many days remain until a production threshold?
- how the spectral trajectory changes across the batch?

This is the same monitoring logic I would later apply to tissue-engineered constructs.

## Why this connects to the lab papers I studied

The tissue-engineered cartilage maturity paper uses visible/NIR spectra to predict **GAG** and **DNA**, then uses the predicted GAG/DNA ratio to classify tissue maturity. That paper is powerful, but it uses two culture durations: 7 and 28 days.

The culture-medium paper gets closer to the real monitoring problem. It collects conditioned medium repeatedly during culture and links NIR spectra to biomarkers such as hyaluronan, lactate, and collagen.

My practical translation here is:

```text
repeated spectral measurements over time
→ temporal machine learning
→ real-time process state estimate
```

In a future tissue-engineering version, I would replace penicillin concentration with GAG/DNA, lactate, collagen, or another maturity marker.

In [ ]:
# If this is a fresh notebook environment, install packages first.
# In Kaggle, most of these are already available.
# Uncomment only if needed.

# !pip install -q numpy pandas matplotlib scikit-learn scipy seaborn tensorflow

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Make local src/ import work whether I run from the repo root or from notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

from bioprocess_utils import (
    find_first_csv,
    load_indpensim_subset,
    add_days_to_threshold,
    make_sequence_windows,
)

print("Project root:", PROJECT_ROOT)
print("TensorFlow:", tf.__version__)

## 1. Find the dataset

I support both routes:

- Kaggle input folder
- local `data/` folder

The expected file is usually called:

```text
100_Batches_IndPenSim_V3.csv
```

It is large, so I do not commit it to GitHub.

In [ ]:
search_roots = [
    "/kaggle/input",
    PROJECT_ROOT / "data",
    Path.cwd(),
]

csv_path = find_first_csv(search_roots)
print("Using CSV:", csv_path)
print("File size GB:", round(csv_path.stat().st_size / 1e9, 2))

## 2. Load a manageable subset

The full dataset is big. For this first project, I do not need every batch and every Raman wavelength.

I load:

- a limited number of batches
- every 10th Raman wavelength column
- the main time and product columns

This keeps the notebook light enough to run while still preserving the core temporal-spectral structure.

In [ ]:
df, raman_cols = load_indpensim_subset(
    csv_path,
    max_batches=12,       # increase later if the notebook runs easily
    raman_stride=10,      # use every 10th Raman wavelength to reduce dimensionality
    max_rows=None         # set e.g. 50000 if your machine struggles
)

print("Data shape:", df.shape)
print("Number of Raman features:", len(raman_cols))
display(df.head())

In [ ]:
print("Columns:")
print(df.columns[:20].tolist())

print("\nBatch IDs:", sorted(df["batch_id"].unique())[:15])
print("Day range:", df["day"].min(), "to", df["day"].max())
print("Penicillin range:", df["penicillin_g_l"].min(), "to", df["penicillin_g_l"].max())

## 3. Visualise product formation over time

I first plot penicillin concentration against time.

This matters because the model should not just memorise random numbers. I want to see the biological production trajectory.

In [ ]:
plot_dir = PROJECT_ROOT / "outputs" / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(10, 5))
for batch_id, group in df.groupby("batch_id"):
    plt.plot(group["day"], group["penicillin_g_l"], alpha=0.75, label=f"Batch {batch_id}")

plt.xlabel("Batch age (days)")
plt.ylabel("Penicillin concentration (g/L)")
plt.title("Penicillin production trajectory across batches")
plt.grid(True, alpha=0.3)
plt.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.savefig(plot_dir / "penicillin_trajectory_by_batch.png", dpi=160)
plt.show()

## 4. Visualise Raman spectral trajectory

A spectrum is a high-dimensional fingerprint.

Here I plot a heatmap for one batch:

- x-axis = Raman wavelengths/features
- y-axis = time
- colour = spectral intensity

If this were tissue engineering, this heatmap would be the spectral fingerprint of a construct or culture medium changing over days.

In [ ]:
example_batch = sorted(df["batch_id"].unique())[0]
one_batch = df[df["batch_id"] == example_batch].sort_values("time_h")

spectral_matrix = one_batch[raman_cols].values

plt.figure(figsize=(12, 6))
plt.imshow(
    spectral_matrix,
    aspect="auto",
    interpolation="nearest",
    cmap="viridis"
)
plt.colorbar(label="Raman intensity")
plt.xlabel("Raman feature index")
plt.ylabel("Time index")
plt.title(f"Raman spectral trajectory heatmap: batch {example_batch}")
plt.tight_layout()
plt.savefig(plot_dir / "raman_heatmap_example_batch.png", dpi=160)
plt.show()

## 5. PCA trajectory of Raman spectra

PCA compresses the Raman spectra into two dimensions.

This is not the model. It is an inspection tool.

I use it to ask:

> Do spectra move through feature space as the batch ages?

If the answer is yes, then predicting time or maturity from spectra is at least plausible.

In [ ]:
pca_sample = df.sample(min(len(df), 5000), random_state=42).copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_sample[raman_cols])

pca = PCA(n_components=2, random_state=42)
pcs = pca.fit_transform(X_scaled)

pca_sample["PC1"] = pcs[:, 0]
pca_sample["PC2"] = pcs[:, 1]

plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    pca_sample["PC1"],
    pca_sample["PC2"],
    c=pca_sample["day"],
    cmap="viridis",
    s=12,
    alpha=0.8
)
plt.colorbar(scatter, label="Batch age (days)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA of Raman spectra coloured by batch age")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(plot_dir / "raman_pca_by_day.png", dpi=160)
plt.show()

print("Explained variance:", pca.explained_variance_ratio_)

## 6. Create the temporal target: days to threshold

I define a practical threshold:

```text
80% of each batch's maximum penicillin concentration
```

For each time point, I calculate how many days remain until that threshold is first reached.

In tissue engineering, this could become:

```text
days until GAG/DNA reaches target maturity threshold
```

In [ ]:
df2 = add_days_to_threshold(df, threshold_fraction=0.80)

print("New shape:", df2.shape)
display(df2[["batch_id", "day", "penicillin_g_l", "threshold_value", "days_to_threshold"]].head())

plt.figure(figsize=(9, 5))
plt.hist(df2["days_to_threshold"], bins=30)
plt.xlabel("Days remaining until 80% production threshold")
plt.ylabel("Count")
plt.title("Distribution of days-to-threshold target")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(plot_dir / "days_to_threshold_distribution.png", dpi=160)
plt.show()

## 7. Prepare features

I use two feature sets:

1. Raman-only features
2. Raman + current penicillin value

In a real online setting, the current product concentration may not always be available. But the notebook keeps it visible so I can compare what spectral-only and mixed-input monitoring might look like.

In [ ]:
feature_cols = raman_cols.copy()

# I keep the target concentration out of the day estimator first.
X = df2[feature_cols].values
y_day = df2["day"].values
y_days_left = df2["days_to_threshold"].values
groups = df2["batch_id"].values

# Group split prevents the same batch leaking into train and test.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y_day, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_day_train, y_day_test = y_day[train_idx], y_day[test_idx]
y_left_train, y_left_test = y_days_left[train_idx], y_days_left[test_idx]

print("Train rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Train batches:", sorted(set(groups[train_idx])))
print("Test batches:", sorted(set(groups[test_idx])))

## 8. Baseline model: Ridge regression

Ridge regression is simple and useful.

If this simple model works, that means the spectra carry strong time information. If it fails badly, the deep model has a harder job.

In [ ]:
day_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

day_ridge.fit(X_train, y_day_train)
pred_day_ridge = day_ridge.predict(X_test)

mae = mean_absolute_error(y_day_test, pred_day_ridge)
rmse = mean_squared_error(y_day_test, pred_day_ridge, squared=False)
r2 = r2_score(y_day_test, pred_day_ridge)

print(f"Ridge day estimator | MAE: {mae:.3f} days | RMSE: {rmse:.3f} | R2: {r2:.3f}")

## 9. Nonlinear model: Random Forest

Random Forest can capture nonlinear relations between spectral features and process time.

I do not treat it as magic. It is just a stronger baseline.

In [ ]:
day_rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    random_state=42,
    n_jobs=-1
)

day_rf.fit(X_train, y_day_train)
pred_day_rf = day_rf.predict(X_test)

mae = mean_absolute_error(y_day_test, pred_day_rf)
rmse = mean_squared_error(y_day_test, pred_day_rf, squared=False)
r2 = r2_score(y_day_test, pred_day_rf)

print(f"Random Forest day estimator | MAE: {mae:.3f} days | RMSE: {rmse:.3f} | R2: {r2:.3f}")

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_day_test, pred_day_rf, s=12, alpha=0.6)
lo = min(y_day_test.min(), pred_day_rf.min())
hi = max(y_day_test.max(), pred_day_rf.max())
plt.plot([lo, hi], [lo, hi], "r--", label="Perfect prediction")
plt.xlabel("Actual batch age (days)")
plt.ylabel("Predicted batch age (days)")
plt.title("Real-time day estimation from Raman spectra")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(plot_dir / "actual_vs_predicted_day_rf.png", dpi=160)
plt.show()

## 10. Predict days remaining until threshold

This is the more useful process-monitoring task.

Instead of asking “what day is it?”, I ask:

> how much time is left before this batch reaches a target production state?

That is close to the tissue-engineering question:

> how long until this construct reaches a target maturity state?

In [ ]:
left_rf = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    random_state=42,
    n_jobs=-1
)

left_rf.fit(X_train, y_left_train)
pred_left_rf = left_rf.predict(X_test)

mae = mean_absolute_error(y_left_test, pred_left_rf)
rmse = mean_squared_error(y_left_test, pred_left_rf, squared=False)
r2 = r2_score(y_left_test, pred_left_rf)

print(f"Days-to-threshold RF | MAE: {mae:.3f} days | RMSE: {rmse:.3f} | R2: {r2:.3f}")

plt.figure(figsize=(7, 6))
plt.scatter(y_left_test, pred_left_rf, s=12, alpha=0.6)
lo = min(y_left_test.min(), pred_left_rf.min())
hi = max(y_left_test.max(), pred_left_rf.max())
plt.plot([lo, hi], [lo, hi], "r--", label="Perfect prediction")
plt.xlabel("Actual days to threshold")
plt.ylabel("Predicted days to threshold")
plt.title("Predicting days remaining until production threshold")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(plot_dir / "actual_vs_predicted_days_to_threshold_rf.png", dpi=160)
plt.show()

## 11. Build sequence windows for LSTM

Now I add a real temporal model.

A single spectrum tells me the current snapshot. A sequence tells me the recent direction.

I build windows like:

```text
last 12 measurements → current day
```

The LSTM learns from the order of measurements.

In [ ]:
# For the LSTM, I use Raman features plus current penicillin concentration.
# Including the process target as an input is not always allowed in real deployment,
# but here it helps demonstrate temporal state tracking. Remove it later for strict spectra-only monitoring.
sequence_feature_cols = raman_cols + ["penicillin_g_l"]

# Scale features before sequence modelling.
df_seq = df2.copy()
seq_scaler = StandardScaler()
df_seq[sequence_feature_cols] = seq_scaler.fit_transform(df_seq[sequence_feature_cols])

X_seq, y_seq, seq_batches, seq_days = make_sequence_windows(
    df_seq,
    feature_cols=sequence_feature_cols,
    target_col="day",
    window=12,
    stride=2
)

print("X_seq shape:", X_seq.shape)
print("y_seq shape:", y_seq.shape)

In [ ]:
# Group split by batch again. I do not want the model training and testing on the same batch.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=7)
train_idx, test_idx = next(splitter.split(X_seq, y_seq, groups=seq_batches))

X_seq_train, X_seq_test = X_seq[train_idx], X_seq[test_idx]
y_seq_train, y_seq_test = y_seq[train_idx], y_seq[test_idx]

print("Sequence train:", X_seq_train.shape)
print("Sequence test:", X_seq_test.shape)

## 12. Train a small LSTM

I keep the neural network small on purpose.

This is a proof-of-concept, not a leaderboard chase. The point is to show that I understand:

- sequence windows
- temporal memory
- batch-wise train/test split
- prediction of process age from recent spectral trajectory

In [ ]:
tf.keras.backend.clear_session()

lstm_model = models.Sequential([
    layers.Input(shape=(X_seq_train.shape[1], X_seq_train.shape[2])),
    layers.Masking(mask_value=0.0),
    layers.LSTM(64, return_sequences=False),
    layers.Dropout(0.2),
    layers.Dense(32, activation="relu"),
    layers.Dense(1)
])

lstm_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="mae",
    metrics=["mae"]
)

lstm_model.summary()

In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor="val_mae",
    patience=8,
    restore_best_weights=True
)

history = lstm_model.fit(
    X_seq_train,
    y_seq_train,
    validation_split=0.2,
    epochs=60,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["mae"], label="Train MAE")
plt.plot(history.history["val_mae"], label="Validation MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE (days)")
plt.title("LSTM training curve")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(plot_dir / "lstm_training_curve.png", dpi=160)
plt.show()

pred_day_lstm = lstm_model.predict(X_seq_test).ravel()

mae = mean_absolute_error(y_seq_test, pred_day_lstm)
rmse = mean_squared_error(y_seq_test, pred_day_lstm, squared=False)
r2 = r2_score(y_seq_test, pred_day_lstm)

print(f"LSTM day estimator | MAE: {mae:.3f} days | RMSE: {rmse:.3f} | R2: {r2:.3f}")

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_seq_test, pred_day_lstm, s=12, alpha=0.6)
lo = min(y_seq_test.min(), pred_day_lstm.min())
hi = max(y_seq_test.max(), pred_day_lstm.max())
plt.plot([lo, hi], [lo, hi], "r--", label="Perfect prediction")
plt.xlabel("Actual batch age (days)")
plt.ylabel("Predicted batch age (days)")
plt.title("LSTM temporal day estimation")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(plot_dir / "actual_vs_predicted_day_lstm.png", dpi=160)
plt.show()

## 13. Save the LSTM model

I save the model so this becomes a real project, not just notebook screenshots.

In a real-time monitoring system, this model would sit behind a streaming pipeline:

```text
new spectrum arrives → preprocess → predict process day / days-to-threshold
```

In [ ]:
model_dir = PROJECT_ROOT / "models"
model_dir.mkdir(parents=True, exist_ok=True)

model_path = model_dir / "lstm_day_estimator.keras"
lstm_model.save(model_path)

print("Saved model:", model_path)

## 14. What I learned

The important lesson is not only the model score.

The important lesson is the workflow:

- repeated spectra form a trajectory
- process time can be learned from spectral changes
- future/threshold state can be framed as a regression problem
- LSTM models need windowed sequences, not shuffled rows
- group-wise splitting prevents fake performance from batch leakage

If I later get real tissue-engineering spectral data, I now know how to turn it into a temporal monitoring problem.

In [ ]:
print("Notebook complete.")
print("Saved plots:")
for p in sorted(plot_dir.glob("*.png")):
    print("-", p)